# Clase: Gestión de Memoria y Optimización en CUDA
Este notebook contiene fragmentos de código organizados para el aprendizaje de CUDA, desde la gestión en CPU hasta optimizaciones avanzadas con memoria compartida.

## 1. Suma de los elementos de un vector: Función Principal (main)
**Explicación:** Este código se ejecuta en la CPU y se encarga de la gestión general: solicita la cantidad de elementos, genera un arreglo aleatorio, mide el tiempo de ejecución de la suma secuencial en CPU y la compara con la ejecución en GPU, imprimiendo finalmente los resultados y tiempos.

In [ ]:
%%writefile main.cu
int main(int argc, char* argv[]) {
    double t_ini, t_fin;
    double time_generateData, time_cpu_seconds, time_gpu_seconds;
    double *A;
    double sum_total_cpu, sum_total_gpu;
    long int n=500000000;

    printf("De cuantos elementos son los arreglos\n");
    scanf("%ld", &n);

    t_ini = clock();
    A = generateRandomArray(n);
    t_fin = clock();
    time_generateData = (t_fin - t_ini) / CLOCKS_PER_SEC;

    t_ini = clock();
    sum_total_cpu = sumOfArraySeq(A, n);
    t_fin = clock();
    time_cpu_seconds = (t_fin - t_ini) / CLOCKS_PER_SEC;

    t_ini = clock();
    sum_total_gpu = sumOfArrayGPU(A,n);
    t_fin = clock();
    time_gpu_seconds = (t_fin - t_ini) / CLOCKS_PER_SEC;

    printf("La suma del arreglo en CPU es: %lf \n", sum_total_cpu);
    printf("La suma del arreglo en GPU es: %lf \n", sum_total_gpu);
    printf("Tiempo para generar datos: %lf segundos.\n", time_generateData);
    printf("Tiempo de procesamiento en CPU: %lf segundos.\n",time_cpu_seconds);
    printf("Tiempo de procesamiento en GPU: %lf segundos.\n", time_gpu_seconds);

    free(A);
}

## 2. Función de envoltura para GPU (sumOfArrayGPU)
**Explicación:** Esta función prepara el entorno para que el Kernel pueda trabajar. Realiza cuatro pasos críticos: reserva memoria en la GPU (`cudaMalloc`), copia los datos del CPU a la GPU (`cudaMemcpyHostToDevice`), lanza la ejecución del Kernel configurando los bloques e hilos, y finalmente copia el resultado de vuelta al CPU para liberar la memoria.

In [ ]:
double sumOfArrayGPU(double *A, long int n){
    double *d_A;
    double *d_sum_total;
    double sum_total;

    //1. Crear memoria en la GPU
    cudaMalloc(&d_sum_total, sizeof(double));
    cudaMalloc(&d_A, n * sizeof(double));

    //Inicializamos en cero
    cudaMemset(d_sum_total, 0, sizeof(double));

    //2. Copiar memoria (CPU-->GPU)
    cudaMemcpy(d_A, A, n * sizeof(double), cudaMemcpyHostToDevice);

    //3. Ejecutar función Kernel
    sumOfArrayKernel <<<(n+TPB-1)/TPB,TPB >>> (d_sum_total,d_A,n);

    //4. Copiar memoria (GPU-->CPU)
    cudaMemcpy(&sum_total, d_sum_total, sizeof(double), cudaMemcpyDeviceToHost);

    cudaFree(d_sum_total);
    cudaFree(d_A);
    cudaDeviceReset();
    return(sum_total);
}

## 3. Kernel de Suma V1 (Memoria Compartida y Función Atómica)
**Explicación:** En esta versión, cada bloque suma sus elementos internamente usando memoria compartida (`__shared__`). Primero, los hilos cargan los datos de la memoria global a la compartida y se sincronizan. Luego, un solo hilo por bloque suma esos valores y utiliza `atomicAdd` para agregar el total del bloque a la variable global de forma segura.

In [ ]:
#define TPB 1024
#define ATOMIC 1 // 0 para no usar el atomicAdd

__global__ void sumOfArrayKernel(double *d_sum_total, double *d_A, long int n) {
    const long int idx = threadIdx.x + blockDim.x * blockIdx.x;
    const int s_idx = threadIdx.x;
    __shared__ double s_data[TPB];

    s_data[s_idx] = (idx < n) ? d_A[idx] : 0.0;
    __syncthreads();

    if (s_idx == 0) {
        double blockSum = 0.0;
        for (int j = 0; j < blockDim.x; j++) {
            blockSum += s_data[j];
        }
        if (ATOMIC) {
            atomicAdd(d_sum_total, blockSum);
        } else {
            *d_sum_total += blockSum; // Resultados no esperados por colisión
        }
    }
}

## 4. Kernel de Suma V2 (Reducción en Memoria Compartida)
**Explicación:** Esta versión optimiza la suma dentro del bloque mediante una reducción paralela. En lugar de que un solo hilo recorra todo el arreglo de la memoria compartida, los hilos colaboran dividiendo el trabajo a la mitad en cada iteración de un ciclo `for`, mejorando drásticamente el rendimiento.

In [ ]:
__global__ void sumOfArrayKernel_V2(double* d_sum_total, double* d_A, long int n) {
    const long int idx = threadIdx.x + blockDim.x * blockIdx.x;
    double blockSum = 0.0;
    const int s_idx = threadIdx.x;
    __shared__ double s_data[TPB];

    s_data[s_idx] = blockSum = (idx < n) ? d_A[idx] : 0.0;
    __syncthreads();

    for (unsigned int s = blockDim.x / 2; s > 0; s >>= 1) {
        if (s_idx < s) {
            s_data[s_idx] = blockSum = blockSum + s_data[s_idx + s];
        }
        __syncthreads();
    }

    if (s_idx == 0) {
        if (ATOMIC) {
            atomicAdd(d_sum_total, blockSum);
        } else {
            *d_sum_total += blockSum;
        }
    }
}

## 5. Producto Punto (Kernel y Lanzador)
**Explicación:** Similar a la suma, este código calcula el producto de elementos correspondientes de dos vectores y los suma. Utiliza memoria compartida para almacenar los productos intermedios de cada hilo antes de realizar la suma final del bloque.

In [ ]:
#define TPB 64
#define ATOMIC 1 

__global__ void dotKernel(int *d_res, const int *d_a, const int *d_b, int n) {
    const int idx = threadIdx.x + blockDim.x * blockIdx.x;
    if (idx >= n) return;
    const int s_idx = threadIdx.x;

    __shared__ int s_prod[TPB];
    s_prod[s_idx] = d_a[idx] * d_b[idx];
    __syncthreads(); // Sincroniza hilos del mismo bloque

    if (s_idx == 0) {
        int blockSum = 0;
        for (int j = 0; j < blockDim.x; ++j) {
            blockSum += s_prod[j];
        }
        if (ATOMIC) {
            atomicAdd(d_res, blockSum);
        } else {
            *d_res += blockSum;
        }
    }
}

void dotLauncher(int *res, const int *a, const int *b, int n) {
    int *d_res, *d_a = 0, *d_b = 0;
    cudaMalloc(&d_res, sizeof(int));
    cudaMalloc(&d_a, n * sizeof(int));
    cudaMalloc(&d_b, n * sizeof(int));

    cudaMemset(d_res, 0, sizeof(int));
    cudaMemcpy(d_a, a, n * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, n * sizeof(int), cudaMemcpyHostToDevice);

    dotKernel << <(n + TPB - 1) / TPB, TPB >> >(d_res, d_a, d_b, n);
    cudaMemcpy(res, d_res, sizeof(int), cudaMemcpyDeviceToHost);

    cudaFree(d_res); cudaFree(d_a); cudaFree(d_b);
}

## 6. Multiplicación de Matrices con Memoria Compartida
**Explicación:** Este es un ejemplo avanzado donde las matrices se dividen en sub-bloques. Cada hilo carga un elemento de una sub-matriz a la memoria compartida (`As` y `Bs`). Esto reduce el acceso a la memoria global, ya que los datos cargados en la memoria compartida son reutilizados por múltiples hilos del mismo bloque para realizar productos punto acumulativos.

In [ ]:
__global__ void Multiplica_Matrices_SM(float *C, float *A, float *B, int nfil, int ncol) {
    // Índices de Bloques e Hilos
    int bx = blockIdx.x; int by = blockIdx.y;
    int tx = threadIdx.x; int ty = threadIdx.y;

    // Índices y pasos para iterar sub-matrices
    int aBegin = ncol * BLOCK_SIZE * by;
    int aEnd = aBegin + ncol - 1;
    int aStep = BLOCK_SIZE;
    int bBegin = BLOCK_SIZE * bx;
    int bStep = BLOCK_SIZE * ncol;

    float sum_sub = 0.0f;

    for (int a = aBegin, b = bBegin; a <= aEnd; a += aStep, b += bStep) {
        __shared__ float As[BLOCK_SIZE][BLOCK_SIZE];
        __shared__ float Bs[BLOCK_SIZE][BLOCK_SIZE];

        // Carga de memoria global a compartida
        As[ty][tx] = A[a + ncol * ty + tx];
        Bs[ty][tx] = B[b + ncol * ty + tx];
        __syncthreads();

        // Multiplicación de sub-matrices
        #pragma unroll
        for (int k = 0; k < BLOCK_SIZE; k++)
            sum_sub += As[ty][k] * Bs[k][tx];
        __syncthreads();
    }

    // Escritura del resultado final en memoria global
    int c = ncol * BLOCK_SIZE * by + BLOCK_SIZE * bx;
    C[c + ncol * ty + tx] = sum_sub;
}